# 📤 Export CSV du modèle en étoile

## Objectifs
1. Exporter toutes les dimensions et la table de faits en CSV
2. Structurer les exports pour Power BI / Excel
3. Générer un fichier ZIP avec tous les exports

## Tables à exporter
### Dimensions de référence
* DimPays, DimMagasin, DimClient, DimVendeur
* DimProduit, DimFamilleProduit, DimMarque
* DimFournisseur, DimModele, DimFacade

### Dimension temporelle
* DimDevis

### Table de faits
* FactLignesDevis

In [0]:
from pyspark.sql import functions as F

# Configuration
CATALOG = "kitchen-proline-data"
SCHEMA = "sales"
EXPORT_PATH = "/Volumes/kitchen-proline-data/raw/exports/csv/"

# Liste des tables à exporter
TABLES = [
    # Dimensions de référence
    "DimPays",
    "DimMagasin",
    "DimClient",
    "DimVendeur",
    "DimProduit",
    "DimFamilleProduit",
    "DimMarque",
    "DimFournisseur",
    "DimModele",
    "DimFacade",
    # Dimension temporelle
    "DimDevis",
    # Table de faits
    "FactLignesDevis"
]

print(f"✓ Configuration chargée")
print(f"📁 Répertoire d'export : {EXPORT_PATH}")
print(f"📊 Nombre de tables à exporter : {len(TABLES)}")

In [0]:
def export_table_to_csv(table_name, export_path):
    """
    Exporte une table Unity Catalog vers un fichier CSV unique
    """
    print(f"\n📤 Export de {table_name}...")
    
    # Lire la table
    full_table_name = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    df = spark.table(full_table_name)
    
    row_count = df.count()
    print(f"   ├─ Lignes : {row_count}")
    print(f"   ├─ Colonnes : {len(df.columns)}")
    
    # Créer le chemin de destination
    output_path = f"{export_path}{table_name}"
    
    # Export en CSV avec un seul fichier (coalesce(1))
    df.coalesce(1).write.mode("overwrite").option("header", "true").option("sep", ",").csv(output_path)
    
    print(f"   └─ ✓ Exporté vers {output_path}")
    return row_count

print("✓ Fonction d'export prête")

## 📦 Export de toutes les tables

In [0]:
print("="*70)
print("EXPORT DES TABLES EN CSV")
print("="*70)

# Dictionnaire pour stocker les statistiques
stats = {}

# Exporter chaque table
for table_name in TABLES:
    try:
        row_count = export_table_to_csv(table_name, EXPORT_PATH)
        stats[table_name] = {"status": "✓", "rows": row_count}
    except Exception as e:
        print(f"   └─ ⚠️ Erreur : {str(e)}")
        stats[table_name] = {"status": "✗", "rows": 0}

print("\n" + "="*70)
print("RÉCAPITULATIF DES EXPORTS")
print("="*70)

# Afficher le récapitulatif
for table_name, stat in stats.items():
    if stat["status"] == "✓":
        print(f"{stat['status']} {table_name:25s} : {stat['rows']:>8,} lignes")
    else:
        print(f"{stat['status']} {table_name:25s} : Échec")

total_rows = sum(s["rows"] for s in stats.values())
print(f"\n📊 Total : {total_rows:,} lignes exportées")
print(f"✓ Tous les exports sont dans : {EXPORT_PATH}")

## 🔍 Vérification des fichiers exportés

In [0]:
print("📁 Fichiers CSV créés :\n")

# Lister les répertoires créés
try:
    directories = dbutils.fs.ls(EXPORT_PATH)
    
    for directory in sorted(directories, key=lambda x: x.name):
        # Chaque export crée un répertoire, lister les fichiers à l'intérieur
        table_name = directory.name.rstrip('/')
        files = dbutils.fs.ls(directory.path)
        
        # Trouver le fichier CSV (ignorer _SUCCESS et autres métadonnées)
        csv_files = [f for f in files if f.name.endswith('.csv')]
        
        if csv_files:
            csv_file = csv_files[0]
            size_mb = csv_file.size / (1024 * 1024)
            print(f"  ├─ {table_name:25s} : {csv_file.name:40s} ({size_mb:.2f} MB)")
        else:
            print(f"  ├─ {table_name:25s} : (fichier CSV non trouvé)")
            
    print(f"\n✓ Tous les exports sont disponibles dans : {EXPORT_PATH}")
except Exception as e:
    print(f"⚠️ Erreur lors de la lecture : {str(e)}")

## 🎯 Prochaines étapes

### Pour Power BI
1. Télécharger les fichiers CSV depuis le volume Unity Catalog
2. Importer dans Power BI Desktop
3. Créer les relations entre les tables (FK numériques vers dimensions) :
   * FactLignesDevis → DimDevis : **IdDevis** → numero_devis
   * FactLignesDevis → DimMagasin : **IdMagasin** → id_magasin
   * FactLignesDevis → DimClient : **IdClient** → id_client
   * FactLignesDevis → DimVendeur : **IdVendeur** → id_vendeur
   * FactLignesDevis → DimPays : **IdPays** → id_pays
   * FactLignesDevis → DimProduit : **IdProduit** → id_produit
   * FactLignesDevis → DimFamilleProduit : **IdFamille** → id_famille
   * FactLignesDevis → DimMarque : **IdMarque** → id_marque
   * FactLignesDevis → DimFournisseur : **IdFournisseur** → id_fournisseur
   * FactLignesDevis → DimModele : **IdModele** → id_modele
   * FactLignesDevis → DimFacade : **IdFacade** → id_facade

### Mesures DAX suggérées
```dax
Chiffre d'Affaires = SUM(FactLignesDevis[prix_net_ht])
Marge Brute = SUM(FactLignesDevis[marge_brute])
Taux de Marge = DIVIDE([Marge Brute], [Chiffre d'Affaires], 0)
Nombre de Devis = DISTINCTCOUNT(FactLignesDevis[IdDevis])
Panier Moyen = DIVIDE([Chiffre d'Affaires], [Nombre de Devis], 0)
```